# 10 · PPML count model

A Poisson (PPML) check on the log-OLS main result: model the establishment
*count* directly with a population offset (a per-capita rate model) and state
clustered errors, which keeps zero-birth counties. **Manuscript: robustness.**

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("../data/cleaned_data_v3.csv")
df = df.rename(columns={
    "pct_highschool_or_more (1990)": "pct_hs_1990",
    "Pop 2010": "pop_2010",
    "Established firms 2022": "firms_2022",
    "Established firms 1989": "firms_1989",
})

for c in ["Index_v2", "income_1989_real_2023", "pct_hs_1990",
          "pop_2010", "firms_2022", "firms_1989"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna(subset=["Index_v2", "income_1989_real_2023", "pct_hs_1990",
                       "pop_2010", "firms_2022", "firms_1989", "State"])
df = df[df["pop_2010"] > 0]

df[["income_1989_scaled", "pct_hs_1990_scaled"]] = StandardScaler().fit_transform(
    df[["income_1989_real_2023", "pct_hs_1990"]])

# offset = log(pop) makes this a per-capita rate model
ppml = smf.glm(
    "firms_2022 ~ Index_v2 + income_1989_scaled + pct_hs_1990_scaled + C(State)",
    data=df, family=sm.families.Poisson(), offset=np.log(df["pop_2010"]),
).fit(cov_type="cluster", cov_kwds={"groups": df["State"]})
print(ppml.summary())

pct = (np.exp(ppml.params["Index_v2"]) - 1) * 100
print(f"\n+1 unit persistence -> {pct:.2f}% change in firms per capita")